# XGBoost, LightGBM, and CatBoost — Specialized Gradient Boosting Libraries

## 1. Shared idea, different engineering choices
All three libraries build boosted tree ensembles, but their training systems differ.

- **XGBoost:** regularized gradient-boosted trees with several tree-building algorithms and explicit regularization controls.
- **LightGBM:** histogram-based learning with optimizations for large tabular data; leaf-wise growth can be fast but needs depth/leaf controls to avoid overfitting.
- **CatBoost:** emphasizes categorical-feature handling and ordered boosting techniques intended to reduce certain forms of target leakage/prediction shift.

These are implementation families, not interchangeable wrappers. Speed and quality depend on dataset, parameterization, hardware and installed version.

## 2. Mathematical foundations
A boosted model is additive: \(F_T(x)=F_0(x)+\sum_{t=1}^{T}\eta f_t(x)\). At each stage, a learner is selected to reduce a loss. Modern implementations often use first- and second-order derivatives and optimize tree structure under regularization.

Common design differences include split search (exact, approximate, histogram), growth strategy (depth-wise/level-wise or leaf-wise variants), categorical feature handling, and regularization/objective implementations.

In [1]:
import importlib.util
import time
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
X, y = make_regression(n_samples=4500, n_features=18, n_informative=11, noise=18, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
available = {name: importlib.util.find_spec(name) is not None for name in ("xgboost", "lightgbm", "catboost")}
print("Optional library availability:", available)


Optional library availability: {'xgboost': True, 'lightgbm': True, 'catboost': True}


## 3. A stable baseline

`HistGradientBoostingRegressor` is included as a scikit-learn baseline so the notebook runs even when the optional packages are not installed. The baseline is not meant to rank the libraries; use the comparison protocol below on your own environment.

In [2]:
benchmarks = []
baseline = HistGradientBoostingRegressor(max_iter=120, learning_rate=0.08, max_leaf_nodes=15, random_state=SEED)
t0 = time.perf_counter(); baseline.fit(X_train, y_train); train_seconds = time.perf_counter()-t0
t0 = time.perf_counter(); pred = baseline.predict(X_test); predict_seconds = time.perf_counter()-t0
benchmarks.append({
    "model":"sklearn HistGradientBoostingRegressor",
    "train_seconds":round(train_seconds,3),
    "predict_seconds":round(predict_seconds,4),
    "test_mse":round(mean_squared_error(y_test,pred),3),
    "test_r2":round(r2_score(y_test,pred),4)
})
print(benchmarks)


[{'model': 'sklearn HistGradientBoostingRegressor', 'train_seconds': 0.135, 'predict_seconds': 0.0039, 'test_mse': 1690.652, 'test_r2': 0.9423}]


## 4. Try each optional library if installed

Install optional packages separately in your own environment if needed. This cell checks availability first and skips unavailable packages without failing the notebook. Parameter support—especially GPU/device flags—varies by library version, so consult the installed version's API before enabling GPU settings.

In [3]:
if available["xgboost"]:
    from xgboost import XGBRegressor
    model = XGBRegressor(
        n_estimators=80, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror",
        n_jobs=2, random_state=SEED
    )
    t0 = time.perf_counter(); model.fit(X_train, y_train); tr = time.perf_counter()-t0
    t0 = time.perf_counter(); p = model.predict(X_test); te = time.perf_counter()-t0
    benchmarks.append({"model":"XGBoost", "train_seconds":round(tr,3), "predict_seconds":round(te,4),
                       "test_mse":round(mean_squared_error(y_test,p),3), "test_r2":round(r2_score(y_test,p),4)})
else:
    print("XGBoost not installed; skipped. Optional install: pip install xgboost")

if available["lightgbm"]:
    from lightgbm import LGBMRegressor
    model = LGBMRegressor(
        n_estimators=80, max_depth=-1, num_leaves=31, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, n_jobs=2, random_state=SEED, verbosity=-1
    )
    t0 = time.perf_counter(); model.fit(X_train, y_train); tr = time.perf_counter()-t0
    t0 = time.perf_counter(); p = model.predict(X_test); te = time.perf_counter()-t0
    benchmarks.append({"model":"LightGBM", "train_seconds":round(tr,3), "predict_seconds":round(te,4),
                       "test_mse":round(mean_squared_error(y_test,p),3), "test_r2":round(r2_score(y_test,p),4)})
else:
    print("LightGBM not installed; skipped. Optional install: pip install lightgbm")

if available["catboost"]:
    from catboost import CatBoostRegressor
    model = CatBoostRegressor(
        iterations=80, depth=6, learning_rate=0.05, loss_function="RMSE",
        verbose=False, random_seed=SEED, allow_writing_files=False
    )
    t0 = time.perf_counter(); model.fit(X_train, y_train); tr = time.perf_counter()-t0
    t0 = time.perf_counter(); p = model.predict(X_test); te = time.perf_counter()-t0
    benchmarks.append({"model":"CatBoost", "train_seconds":round(tr,3), "predict_seconds":round(te,4),
                       "test_mse":round(mean_squared_error(y_test,p),3), "test_r2":round(r2_score(y_test,p),4)})
else:
    print("CatBoost not installed; skipped. Optional install: pip install catboost")

print("\nBenchmark results:")
for row in benchmarks:
    print(row)


/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Benchmark results:
{'model': 'sklearn HistGradientBoostingRegressor', 'train_seconds': 0.135, 'predict_seconds': 0.0039, 'test_mse': 1690.652, 'test_r2': 0.9423}
{'model': 'XGBoost', 'train_seconds': 0.093, 'predict_seconds': 0.0015, 'test_mse': 4013.863, 'test_r2': 0.863}
{'model': 'LightGBM', 'train_seconds': 0.086, 'predict_seconds': 0.0045, 'test_mse': 3458.343, 'test_r2': 0.8819}
{'model': 'CatBoost', 'train_seconds': 0.189, 'predict_seconds': 0.005, 'test_mse': 3566.646, 'test_r2': 0.8782}


## 5. Hyperparameters to understand first

- Shared: number of boosting rounds, learning rate, tree depth/leaves, row subsampling, feature subsampling, loss/objective, regularization and early stopping.
- XGBoost: `min_child_weight`, `gamma`, `reg_alpha`, `reg_lambda`, tree method, and device selection in the installed version.
- LightGBM: `num_leaves`, `max_depth`, `min_child_samples`, `feature_fraction`/`colsample_bytree`, and `bagging_fraction`/`subsample` settings.
- CatBoost: `depth`, `l2_leaf_reg`, categorical feature specification, and ordered/boosting options.
- Tune on a validation protocol; do not compare libraries using wildly different budgets and then attribute all differences to the algorithm.

## 6. Complexity, millions of rows, and memory

All three reduce split search through optimized tree-building strategies, but precise complexity depends on binning, tree growth, number of features, categorical handling, and objective. Memory includes input/binned data, gradients/Hessians, tree structures and histograms. Measure peak resident memory as well as wall-clock time.

For millions of rows:
1. Establish a reliable baseline and fixed validation split.
2. Use sensible feature types and avoid unnecessary one-hot expansion of high-cardinality categoricals.
3. Tune depth/leaves, subsampling, bins and early stopping.
4. Check single-thread vs multi-thread behavior and concurrent jobs.
5. Measure prediction latency and model size, not only training time.

## 7. GPU training
GPU support is library- and version-specific. Verify the installed release's accepted device/tree-method parameters rather than copying a flag from another version. GPU speedups depend on dataset size, feature types, available VRAM, transfer costs, and whether training is compute- or memory-bound. Small tabular datasets may train faster on CPU.

In [4]:
assert len(benchmarks) >= 1
assert all(np.isfinite(row["test_mse"]) and np.isfinite(row["test_r2"]) for row in benchmarks)
assert all(row["train_seconds"] >= 0 and row["predict_seconds"] >= 0 for row in benchmarks)
print("Library comparison notebook checks passed.")


Library comparison notebook checks passed.


## 8. Fair comparison checklist

Compare on the same split and metric; report library versions, parameters, hardware, thread count, training time, inference latency, model size, and memory. Tune each model responsibly, including early stopping where supported. Repeat with multiple seeds when randomness can materially affect results. No single library is the universal winner.